<a href="https://colab.research.google.com/github/Bhavyateja04/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [2]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [3]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [5]:
sample_data = """
Ravi Kumar

Email: ravi.kumar@gmail.com
Phone: +91 9876543210

Education:
B.Tech in Computer Science and Engineering, Aditya College of Engineering and Technology, 2024

Skills:
Python, Java, SQL, Git, Linux, Data Structures

Projects:
Online Library Management System
Student Attendance Tracker

Experience:
1 year internship in Software Development

---

Sneha Reddy

Email: sneha.reddy@gmail.com

Education:
B.Tech in Information Technology, JNTUK, 2025

Skills:
Python, Django, React, HTML, CSS, JavaScript

Projects:
E-Commerce Website
Placement Preparation Portal

Experience:
0.5 years internship in Web Development

---

Arun Pillai

Email: arun.pillai@gmail.com
Phone: +91 9988776655

Education:
B.Tech in Computer Science, VIT Chennai, 2024

Skills:
Java, Spring Boot, MySQL, Docker, Git, REST APIs, Linux, AWS, MongoDB

Projects:
Product Catalog Backend API
Online Banking Management System

Experience:
1 year Software Engineering Internship
"""

with open("sample_resumes.txt", "w", encoding="utf-8") as f:
    f.write(sample_data)

print("sample_resumes.txt created successfully!")

sample_resumes.txt created successfully!


In [7]:
with open('sample_resumes.txt', 'r', encoding='utf-8') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 3 sample résumés

Résumé 1: Ravi Kumar — 6 skills, 1.0 years exp

Résumé 2: Sneha Reddy — 6 skills, 0.5 years exp

Résumé 3: Arun Pillai — 9 skills, 1.0 years exp

=== Full first result ===
{
  "name": "Ravi Kumar",
  "email": "ravi.kumar@gmail.com",
  "phone": "+91 9876543210",
  "education": [
    {
      "degree": "B.Tech in Computer Science and Engineering",
      "institution": "Aditya College of Engineering and Technology",
      "year": 2024
    }
  ],
  "skills": [
    "Python",
    "Java",
    "SQL",
    "Git",
    "Linux",
    "Data Structures"
  ],
  "projects": [
    "Online Library Management System",
    "Student Attendance Tracker"
  ],
  "experience_years": 1.0
}


In [8]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Unexpected success: {"name":"John Doe","email":"john.doe@example.com","phone":"123-456-7890","education":[{"degree":"Master of Science in Computer Science","institution":"Stanford University","year":2022},{"degree":"Bachelor of Science in Electrical Engineering","institution":"University of California, Berkeley","year":2020}],"skills":["Python","Java","C++","Machine Learning","Data Science","Cloud Computing (AWS)","Web Development (React, Node.js)"],"projects":["E-commerce Recommendation System","Natural Language Processing Chatbot"],"experience_years":2.5}


In [11]:
def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:

    if not raw_text or not raw_text.strip():
        raise ValueError("Validation error for resume1")

    for attempt in range(max_retries + 1):
        ...

In [12]:
try:
    bad = extract_resume('')
    print('Unexpected success')
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e))

Caught gracefully: ValueError
Message: Validation error for resume1
